# Notebook 03 - Train and evaluate the models: Objective (a) and (b), RQ1/H1
Thesis Sections 3.3.1-3.3.4, 3.3.6, 3.4.3 and 3.5.1-3.5.2.

1. Trains the **residual 1D-CNN** and the **CNN-BiLSTM with attention** with the class-weighted loss (**Equation 1**), five seeds each.
2. Runs the preliminary **weighted BCE vs focal loss** comparison.
3. Trains the **EBM comparator** on PTB-XL+ features.
4. Evaluates everything **once** on the test fold (fold 10): AUROC, F1, sensitivity, specificity, ECE, Brier, 95% CIs, DeLong tests.

**GPU required:** Runtime > Change runtime type > T4 GPU. One seed of one model takes about 15-40 minutes on a T4. Colab may disconnect after a few hours; just run the notebook again and finished seeds are skipped (they are saved in Drive).

In [ ]:
# STEP 0 - Setup. Run this cell first, every time you open the notebook.
import os, sys, glob, json, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # the branch of your GitHub repository to use

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive" and allow
    PROJECT = "/content/drive/MyDrive/FUTA-MTECH"        # everything you keep lives here
    REPO = "/content/FUTA-MTECH-PROJECT"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/Scientist47/FUTA-MTECH-PROJECT.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip -q install wfdb neurokit2 shap captum interpret-core
else:                                                    # running on your own computer
    PROJECT = os.environ.get("FUTA_PROJECT", os.path.abspath(".."))
    REPO = os.environ.get("FUTA_REPO", os.path.abspath(".."))

sys.path.insert(0, REPO)
os.chdir(REPO)
DATA = f"{PROJECT}/data"            # downloaded and processed data
RESULTS = f"{PROJECT}/results"      # tables, figures and trained models for Chapter 4
for d in (DATA, RESULTS, f"{RESULTS}/tables", f"{RESULTS}/figures", f"{RESULTS}/models"):
    os.makedirs(d, exist_ok=True)

# QUICK_TEST = True runs everything with tiny settings, to check the notebook works.
# Leave it False for your real thesis results.
QUICK_TEST = os.environ.get("FUTA_QUICK_TEST") == "1"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - see the guide, Step 1.3)")
print("Project folder:", PROJECT)

In [ ]:
import yaml
from preprocessing.ptbxl import SUPERCLASSES
from training.train import train, load_trained, predict, macro_auroc
P = f"{DATA}/processed/ptbxl100"
data = {k: np.load(f"{P}/{k}.npy") for k in ["X_train", "y_train", "X_val", "y_val"]}
for k, v in data.items():
    print(k, v.shape)

## Step 3.1 - Training settings
Read from `training/configs/*.yaml` (Adam, learning rate 0.001 halved when validation loss stops falling, batch 64, early stopping after 8 epochs without a better validation macro-AUROC, augmentation on). In a quick test the settings are cut down.

In [ ]:
configs = {name: yaml.safe_load(open(f"training/configs/ptbxl_{name}.yaml"))
           for name in ["cnn1d", "cnn_bilstm_attention"]}
SEEDS = configs["cnn1d"]["seeds"]
if QUICK_TEST:
    SEEDS = [0, 1]
    for c in configs.values():
        c.update(max_epochs=2, patience=1)
configs["cnn1d"]

## Step 3.2 - Train both architectures, five seeds each
Each line printed is one epoch: training loss, validation loss (Equation 1) and validation macro-AUROC. The best epoch is saved to Drive.

In [ ]:
summaries = {}
for arch, cfg in configs.items():
    for seed in SEEDS:
        run_dir = f"{RESULTS}/models/ptbxl_{arch}/seed{seed}"
        if os.path.exists(f"{run_dir}/summary.json"):
            print(f"{arch} seed {seed}: already trained, skipping")
        else:
            print(f"===== {arch}, seed {seed} =====")
            train(cfg, seed, data, out_dir=f"{RESULTS}/models/ptbxl_{arch}")
        summaries[(arch, seed)] = json.load(open(f"{run_dir}/summary.json"))
pd.DataFrame(summaries).T[["best_epoch", "best_val_macro_auroc"]]

**Expected on the real data:** validation macro-AUROC around 0.90-0.93 for both architectures (the benchmark is about 0.93).

## Step 3.3 - Learning curves (appendix figure)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, arch in zip(axes, configs):
    for seed in SEEDS:
        h = pd.DataFrame(json.load(open(f"{RESULTS}/models/ptbxl_{arch}/seed{seed}/history.json")))
        ax.plot(h.epoch, h.val_macro_auroc, label=f"seed {seed}")
    ax.set_title(arch); ax.set_xlabel("epoch"); ax.set_ylabel("validation macro-AUROC"); ax.legend()
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/learning_curves.png", dpi=300); plt.show()

## Step 3.4 - Preliminary comparison: weighted BCE vs focal loss (Section 3.3.4)
One seed of the 1D-CNN with focal loss, compared on the **validation** fold (fold 9), so the test fold stays untouched.

In [ ]:
focal_cfg = dict(configs["cnn1d"], loss="focal")
fdir = f"{RESULTS}/models/ptbxl_cnn1d_focal"
if not os.path.exists(f"{fdir}/seed0/summary.json"):
    train(focal_cfg, 0, data, out_dir=fdir)
rows = {}
for name, d in [("Weighted BCE (Eq. 1)", f"{RESULTS}/models/ptbxl_cnn1d/seed0"), ("Focal loss", f"{fdir}/seed0")]:
    m, th = load_trained("cnn1d", 5, d)
    p = predict(m, data["X_val"])
    rows[name] = {"val macro-AUROC": macro_auroc(data["y_val"], p),
                  **{f"AUROC {c}": macro_auroc(data["y_val"][:, [i]], p[:, [i]]) for i, c in enumerate(SUPERCLASSES)}}
loss_table = pd.DataFrame(rows).T.round(3)
loss_table.to_csv(f"{RESULTS}/tables/table4_x_loss_comparison_val.csv"); loss_table

## Step 3.5 - EBM comparator on PTB-XL+ features (Section 3.3.6)
One Explainable Boosting Machine per superclass, trained on the 12SL measured features for folds 1-8. Missing values are filled with the training median. Takes 10-30 minutes on CPU.

In [ ]:
from models.ebm_comparator import load_ptbxl_plus_features, impute, train_ebm, predict_ebm
from training.train import choose_thresholds
import pickle
paths = json.load(open(f"{PROJECT}/paths.json"))
ids = {k: np.load(f"{P}/ecg_id_{k}.npy") for k in ["train", "val", "test"]}
if QUICK_TEST:      # no PTB-XL+ in a quick test: use simple made-up features instead
    rng = np.random.default_rng(0)
    feats = {k: pd.DataFrame(np.load(f"{P}/X_{k}.npy")[:, :, ::50].reshape(len(ids[k]), -1)) for k in ids}
    feats = {k: v.set_axis([f"f{i}" for i in range(v.shape[1])], axis=1) for k, v in feats.items()}
else:
    PTBXL_PLUS = paths["PTBXL_PLUS"]
    if IN_COLAB and not os.path.exists(PTBXL_PLUS):
        !unzip -q -o "{DATA}/ptb-xl-plus.zip" -d /content/ptb-xl-plus
    feats = {k: load_ptbxl_plus_features(PTBXL_PLUS, ids[k]) for k in ids}
    feats = {k: v[feats["train"].columns] for k, v in feats.items()}
feats["train"], feats["val"], feats["test"] = impute(feats["train"], feats["val"], feats["test"])
print("EBM features:", feats["train"].shape[1])
ebm_path = f"{RESULTS}/models/ebm_ptbxl.pkl"
if os.path.exists(ebm_path):
    ebms = pickle.load(open(ebm_path, "rb"))
else:
    ebms = train_ebm(feats["train"], data["y_train"], SUPERCLASSES,
                     **({"max_rounds": 50, "interactions": 0} if QUICK_TEST else {}))
    pickle.dump(ebms, open(ebm_path, "wb"))
p_val_ebm = predict_ebm(ebms, feats["val"])
ebm_thresholds = choose_thresholds(data["y_val"], p_val_ebm)
print("EBM validation macro-AUROC:", round(macro_auroc(data["y_val"], p_val_ebm), 3))

In [ ]:
# EBM shape functions: the comparator's own explanation (one figure per superclass)
ebm = ebms["MI"]
glob_exp = ebm.explain_global()
imp = pd.Series(ebm.term_importances(), index=ebm.term_names_).sort_values(ascending=False)
imp.head(15).iloc[::-1].plot.barh(figsize=(7, 5), title="EBM for MI: 15 most important features")
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/ebm_MI_importance.png", dpi=300); plt.show()

## Step 3.6 - FINAL evaluation on the test fold (fold 10). Run once.
Your methodology says fold 10 is used **only once**, after the hypotheses (H1-H3) and analysis plan are fixed (Section 3.6). Change the next line to `True` only when you are ready. Thresholds were already fixed on fold 9 during training.

In [ ]:
I_HAVE_FIXED_MY_HYPOTHESES = False or QUICK_TEST
assert I_HAVE_FIXED_MY_HYPOTHESES, "Set this to True only for the final, one-time test evaluation."
X_test, y_test = np.load(f"{P}/X_test.npy"), np.load(f"{P}/y_test.npy")
from evaluation.classification_metrics import (compute_metrics, metrics_with_ci, summarise_seeds,
                                               delong_test, paired_bootstrap_auc_diff)
probs, per_seed = {}, []
for arch in configs:
    ps = []
    for seed in SEEDS:
        m, th = load_trained(arch, 5, f"{RESULTS}/models/ptbxl_{arch}/seed{seed}")
        p = predict(m, X_test); ps.append(p)
        t = compute_metrics(y_test, p, th, SUPERCLASSES)
        t["arch"], t["seed"] = arch, seed
        per_seed.append(t.reset_index().rename(columns={"index": "class"}))
    probs[arch] = np.mean(ps, axis=0)        # average of the five seeds, used for the paired tests
per_seed = pd.concat(per_seed)
per_seed.to_csv(f"{RESULTS}/tables/test_metrics_every_seed.csv", index=False)
probs["ebm"] = predict_ebm(ebms, feats["test"])
np.savez(f"{RESULTS}/test_probabilities.npz", y_test=y_test, **probs)

### Table 4.2 - Macro-AUROC by architecture (mean of the seeds, 95% CI)

In [ ]:
rows = {}
for arch in configs:
    v = per_seed[(per_seed.arch == arch) & (per_seed["class"] == "Macro")]["AUROC"].to_numpy()
    m, lo, hi = summarise_seeds(v)
    rows[arch] = {"Macro-AUROC": f"{m:.3f} ({lo:.3f}-{hi:.3f})", "Seeds": len(v)}
ebm_m = compute_metrics(y_test, probs["ebm"], ebm_thresholds, SUPERCLASSES).loc["Macro", "AUROC"]
rows["EBM comparator"] = {"Macro-AUROC": f"{ebm_m:.3f}", "Seeds": 1}
rows["Benchmark (Strodthoff et al., 2021)"] = {"Macro-AUROC": "about 0.93", "Seeds": "-"}
table42 = pd.DataFrame(rows).T
table42.to_csv(f"{RESULTS}/tables/table4_2_macro_auroc.csv"); table42

### Table 4.3 - Per-class metrics (mean across seeds, 95% CI)

In [ ]:
out = []
for arch in configs:
    sub = per_seed[per_seed.arch == arch]
    for cls in SUPERCLASSES:
        r = {"Model": arch, "Class": cls}
        for metric in ["AUROC", "F1", "Sensitivity", "Specificity", "ECE", "Brier"]:
            m, lo, hi = summarise_seeds(sub[sub["class"] == cls][metric].to_numpy())
            r[metric] = f"{m:.3f} ({lo:.3f}-{hi:.3f})"
        out.append(r)
e = compute_metrics(y_test, probs["ebm"], ebm_thresholds, SUPERCLASSES)
for cls in SUPERCLASSES:
    out.append({"Model": "EBM", "Class": cls, **{k: f"{e.loc[cls, k]:.3f}" for k in e.columns}})
table43 = pd.DataFrame(out)
table43.to_csv(f"{RESULTS}/tables/table4_3_per_class_metrics.csv", index=False); table43

### Table 4.4 - Paired comparisons (DeLong test and paired bootstrap), per class

In [ ]:
rows = []
pairs = [("cnn1d", "cnn_bilstm_attention"), ("cnn1d", "ebm"), ("cnn_bilstm_attention", "ebm")]
for a, b in pairs:
    for c, cls in enumerate(SUPERCLASSES):
        auc_a, auc_b, z, p = delong_test(y_test[:, c], probs[a][:, c], probs[b][:, c])
        d, lo, hi, pb = paired_bootstrap_auc_diff(y_test[:, c], probs[a][:, c], probs[b][:, c],
                                                  n_boot=200 if QUICK_TEST else 2000)
        rows.append({"Comparison": f"{a} vs {b}", "Class": cls, "AUROC A": round(auc_a, 3),
                     "AUROC B": round(auc_b, 3), "Difference (95% CI)": f"{d:.3f} ({lo:.3f} to {hi:.3f})",
                     "DeLong p": round(p, 4), "Bootstrap p": round(pb, 4)})
table44 = pd.DataFrame(rows)
table44.to_csv(f"{RESULTS}/tables/table4_4_model_comparisons.csv", index=False); table44

### Figure 4.3 - ROC curves and calibration (seed-averaged predictions)

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for c, cls in enumerate(SUPERCLASSES):
    for name, p in probs.items():
        fpr, tpr, _ = roc_curve(y_test[:, c], p[:, c])
        axes[0, c].plot(fpr, tpr, label=name)
        fr, mp = calibration_curve(y_test[:, c], p[:, c], n_bins=10)
        axes[1, c].plot(mp, fr, marker="o", ms=3, label=name)
    axes[0, c].plot([0, 1], [0, 1], "k:"); axes[0, c].set_title(f"ROC - {cls}")
    axes[1, c].plot([0, 1], [0, 1], "k:"); axes[1, c].set_title(f"Calibration - {cls}")
axes[0, 0].legend(); axes[1, 0].set_xlabel("predicted probability"); axes[1, 0].set_ylabel("observed frequency")
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/fig4_3_roc_calibration.png", dpi=300); plt.show()

## Step 3.7 - External validation on PhysioNet/CinC 2021 (Section 3.5.2, optional if time runs short)
The CinC 2021 training set is large (tens of GB). Download only the non-PTB sources you need into Drive, e.g. from a terminal: `wget -r -N -c -np https://physionet.org/files/challenge-2021/1.0.3/training/georgia/`. Then set `CINC_ROOT` below and run. The PTB-XL model is **not retrained**; the PTB-XL training statistics are reused for z-scoring.

In [ ]:
CINC_ROOT = f"{DATA}/cinc2021/training"      # folder containing the source folders
if os.path.exists(CINC_ROOT) and not QUICK_TEST:
    from preprocessing.cinc2021 import build_snomed_map, load_external
    from preprocessing.filters import clean, apply_zscore
    from evaluation.external_validation import evaluate_external
    smap = build_snomed_map(paths["PTBXL_PLUS"], paths["PTBXL"])
    Xe, ye, info = load_external(CINC_ROOT, smap)
    print(info.source.value_counts())
    Xe = apply_zscore(clean(Xe, fs=100), dict(np.load(f"{P}/lead_stats.npz")))
    for arch in configs:
        m, th = load_trained(arch, 5, f"{RESULTS}/models/ptbxl_{arch}/seed0")
        t = evaluate_external(m, Xe, ye, th, SUPERCLASSES)
        t.to_csv(f"{RESULTS}/tables/table4_5_external_{arch}.csv"); display(t)
else:
    print("CinC 2021 data not found - skipped.")

**Done.** For Chapter 4, Section 4.3 keep Tables 4.2-4.4 (and 4.5 if you ran external validation), Figure 4.3 and the loss comparison table. Next: notebook 04.